# Identifying Keywords About Climate Change

First, we will apply Latent Dirichlet Allocation (LDA) on Integovernmental Panel on Climate Change (IPCC) reports to extract keywords about climate change. LDA is a generative probabilistic model and view each document as random mixtures over latent topics, in which each topic is a probaiblity distribution over a fixed vocabulary. 
LDA can discover topics that the documents contain and the extent to which these topics are included in each document. Thus, it identifies words that are highly present in a large number of IPCC reports. 

## IPCC Reports

Reports from IPCC provide a comprehensive summary of the drivers of climate change, its impacts, future risks and how adaptation and mitigation can reduce those risks. The in-depth description of climate change that this source provides helps us to extract seed words of climate risks. We include Synthesis reports, which provide a comprehensive and integrated view, as well as Special reports, which focus on one specific issue of climate change. All reports can be downloaded from the IPCC website as PDF files and transferred to text files using free online conversion tool.


## LDA from Scratch

The basic idea of LDA is that documents are represented as random mixtures over latent topics, where each topics is represented as a distribution of words. We will first use it to extract seed words describing climate change from IPCC reports.

To better understand how LDA works, we will first implement it from scratch in Python. 

In [ ]:
# To be run the first time in order to install the libraries
!pip install nltk
!pip install wordcloud

In [6]:
# Importing libraries
import random
from collections import Counter
import nltk
import matplotlib.pyplot as plt
import re
from re import RegexFlag
from wordcloud import WordCloud

### Preprocessing

We take an example of few sentences as documents. We first need to preprocess those sentences by dropping punctuation, lowercasing the words, dropping stopwords. 

In [7]:
random.seed(0)  
    
# Collection of documents. 
documents = [
    ['The sky is blue and beautiful.'],
    ['Love this blue and beautiful sky!'],        
    ['The quick brown fox jumps over the lazy dog.'],        
    ["A king's breakfast has sausages, ham, bacon, eggs, toast and beans"],        
    ['I love green eggs, ham, sausages and bacon!'],        
    ['The brown fox is quick and the blue dog is lazy!'],        
    ['The sky is very blue and the sky is very beautiful today'],        
    ['The dog is lazy but the brown fox is quick!']        
]

topic_names = ['food', 'weather', 'animals']

In [8]:
# We first preprocess the documents by removing special characters, extra whitespaces, digits and stopwords.
def pre_process_documents(doc):
    wpt = nltk.WordPunctTokenizer()
    stop_words = nltk.corpus.stopwords.words('english')
    for i in range(len(doc)):            
        # lower case and remove special characters\whitespaces
        doc[i][0] = re.sub(r'[^a-zA-Z\s]', '', doc[i][0], flags=RegexFlag.IGNORECASE | RegexFlag.A)
        doc[i][0] = doc[i][0].lower()
        doc[i][0] = doc[i][0].strip()
        # tokenize document
        tokens = wpt.tokenize(doc[i][0])
        # filter stopwords out of document
        filtered_tokens = [token for token in tokens if token not in stop_words]       
        doc[i] = filtered_tokens
    return filtered_tokens

pre_processed_documents = pre_process_documents(documents)

### Initialization of the Model

We first initialize some variables that will be used later in the model. 

In [12]:
# number of topics 
K = 3

In [23]:
# Initialization

# How many times each topic is assigned to each document.
document_topic_counts = [Counter()
                             for _ in documents]

# How many times each word is assigned to each topic.
topic_word_counts = [Counter() for _ in range(K)]

# The total number of words assigned to each topic.
topic_counts = [0 for _ in range(K)]

# The total number of words contained in each document.
document_lengths = [len(d) for d in documents] # N

# Total number of distinct words
distinct_words = set(word for document in documents for word in document)

# The number of distinct words
V = len(distinct_words)

# The number of documents
D = len(documents) 

# document_topics is a Collection that assign a topic (number between 0 and K-1) to each word in each document.
# For example: document_topic[3][4] -> [4 document][id of topic assigned to 5 word]
# This collection defines each document's distribution over topics, and
# implicitly defines each topic's distribution over words.
document_topics = [[random.randrange(K) for word in document]
                    for document in documents] # random initialization

# Fill the counts based on the random initialization
for d in range(D):
    for word, topic in zip(documents[d], document_topics[d]):
        document_topic_counts[d][topic] += 1
        topic_word_counts[topic][word] += 1
        topic_counts[topic] += 1


### Gibbs Sampling

In [ ]:
max_iteration = 1000

In [ ]:
def p_word_given_topic(word, 
                    topic,
                    V, 
                    topic_word_counts,
                    topic_counts,
                    beta=0.1):
    '''
    P(word|topic,Beta)
    The fraction of words assigned to topic
    that equal word (plus some smoothing)
    '''    
    return ((topic_word_counts[topic][word] + beta) / 
            (topic_counts[topic] + V * beta))

In [ ]:
def p_topic_given_document(topic, d,
                        document_topic_counts,
                        document_lengths,
                        alpha=0.1):
    '''
    P(topic|d,Alpha)
    The fraction of words in document d
    that are assigned to topic (plus some smoothing)
    '''    
    return ((document_topic_counts[d][topic] + alpha) / 
            (document_lengths[d] + K * alpha))

In [ ]:
def topic_weight(d, 
                word, 
                topic,
                topic_word_counts,
                topic_counts,
                document_topic_counts,
                document_lengths):
    '''
    P(topic|word,Alpha,Beta) = P(topic|d,Alpha) * P(word|topic,Beta)
    Given a document and a word in that document,
    return the weight for the k-th topic
    '''    
    return p_word_given_topic(word, 
                    topic,
                    V, 
                    topic_word_counts,
                    topic_counts) * p_topic_given_document(topic, d,
                                                            document_topic_counts,
                                                            document_lengths)

In [ ]:
def sample_from_weights(weights):
    '''
    This function randomly choose an index based on an arbitrary set of weights.
    Return the first weight's index that is greater than or equal to a random number.
    '''
    total = sum(weights)
    rnd = total * random.random()  # uniform between 0 and total
    for i, w in enumerate(weights):
        rnd -= w  # return the smallest i such that
        if rnd <= 0: return i  # sum(weights[:(i+1)]) >= rnd

In [ ]:
def choose_new_topic(d, 
                word, 
                topic,
                topic_word_counts,
                topic_counts,
                document_topic_counts,
                document_lengths):
    return sample_from_weights([topic_weight(d, 
                word, 
                topic,
                topic_word_counts,
                topic_counts,
                document_topic_counts,
                document_lengths)
                    for k in range(K)])

In [ ]:
def gibbs_sample(max_iteration,
                document_topic_counts,
                topic_word_counts,
                document_lengths,
                document_topics):
    '''
    Gibbs sampling https://en.wikipedia.org/wiki/Gibbs_sampling.
    '''
    for _ in range(self.max_iteration):
        for d in range(self.D):
            for i, (word, topic) in enumerate(zip(documents[d],
                                                    document_topics[d])):        
                # remove this word / topic from the counts
                # so that it doesn't influence the weights
                document_topic_counts[d][topic] -= 1
                topic_word_counts[topic][word] -= 1
                topic_counts[topic] -= 1
                document_lengths[d] -= 1
    
                # choose a new topic based on the weights
                new_topic = choose_new_topic(d, word)
                document_topics[d][i] = new_topic
    
                # and now add it back to the counts
                document_topic_counts[d][new_topic] += 1
                topic_word_counts[new_topic][word] += 1
                topic_counts[new_topic] += 1
                document_lengths[d] += 1

    return document_topic_counts, topic_word_counts, topic_counts, document_lengths